# TC_12: CNN image perturbation
Apply blur and brightness perturbation. Run full diagnostic pipline and comare against NB10 CNN baseline.
UQ: Deep Ensemble, MC Dropout
XAI: Grad-CAM

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms


sys.path.append('..')
os.chdir('..')
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

from src.utils.config_loader import load_config
from src.utils.data_loader import load_image_data
from src.data_diagnostics.quality_checks import check_image_quality
from src.utils.visualization import plot_class_distribution, plot_correlation, plot_feature_boxplots
from src.utils.performance_tracker import PerformanceTracker
from src.models.image_models import train_simple_cnn, evaluate_image_model
from src.inference_diagnostics.uncertainty import mc_dropout_prediction_img, deep_ensemble_prediction_img
from src.inference_diagnostics.explainability import gradcam_img, collect_gradcam_feedback
from src.inference_diagnostics.flagging import assign_flags, evaluate_flags
from src.utils.visualization import plot_uncertainty_distribution, plot_flag_distribution
from src.utils.report_generator import generate_report,save_report
from src.data_diagnostics.perturbations import apply_blur_img, apply_brightness_img
from src.utils.visualization import plot_uncertainty_distribution, plot_flag_distribution, plot_gradcam_sample
from src.utils.config_loader import load_config, get_image_config, load_threshold
from src.utils.per_sample_logger import save_per_sample, build_experiment_id
from src.inference_diagnostics.explainability import collect_gradcam_feedback_resumable


config = load_config()
tracker = PerformanceTracker()

dataset_short = get_image_config(config)['short_name']
dataset_name = get_image_config(config)['name']
model_name = 'simple_cnn'
review_count = config['stage3_inference']['explainability']['gradcam']['max_samples_to_explain']

In [2]:
class BlurTransform:
    def __init__(self, sigma):
        self.sigma = sigma
    def __call__(self, img):
        return apply_blur_img(img, self.sigma)

class BrightnessTransform:
        def __init__(self, delta):
            self.delta = delta
        def __call__(self, img):
            return apply_brightness_img(img, self.delta)

def load_corrupted_image_data(config, perturbation_type, level):
    dataset_config = get_image_config(config)
    image_size = dataset_config['image_size']
    data_path =  os.path.join(config['paths']['data_raw'], dataset_config['folder_name'])

    transform_list = [transforms.Resize((image_size, image_size))]

    if perturbation_type == 'blur':
        transform_list.append(BlurTransform(sigma = level))
    else:
        transform_list.append(BrightnessTransform(delta = level))

    # grayscale datasets must stay single channel
    if dataset_config['num_channels'] == 1:
        transform_list.append(transforms.Grayscale(num_output_channels=1))

    transform_list.append(transforms.ToTensor())
    corrupted_transform = transforms.Compose(transform_list)

    train_set =  datasets.ImageFolder(
        root = os.path.join(data_path, 'train'),
        transform = corrupted_transform
    )

    return train_set




In [3]:
train_set, test_set = load_image_data(config)
seed = config['random_seeds']['primary_seed']
blur_levels = config['stage1_data_quality']['perturbations']['image_corruption']['blur_sigma']
brightness_level = config['stage1_data_quality']['perturbations']['image_corruption']['brightness_delta']

# Load CNN golden baseline thresholds written by TC_10
mc_threshold = load_threshold(config, dataset_short, model_name, 'mc')
de_threshold = load_threshold(config, dataset_short, model_name, 'de')
print(f"Loaded thresholds - MC: {mc_threshold}, DE: {de_threshold}")

Loading dataset.
Dataset: Fashion-MNIST
Loaded: 60000 train and 10000 test
Image size: 28, Channels: 1
Classes: 10
Class names: ['ankle-boot', 'bag', 'coat', 'dress', 'pullover', 'sandal', 'shirt', 'sneaker', 't-shirt', 'trouser']
Loaded thresholds - MC: 0.0265, DE: 0.0633


### Blur perturbation

In [4]:
blur_result = {}

for sigma in blur_levels:

    # Load corrupted training data
    train_set_corrupted = load_corrupted_image_data(config, 'blur', sigma)

    tracker.start_performance_track()
    cnn_model = train_simple_cnn(train_set_corrupted, config)
    tracker.stop_performance_track(f"CNN training blur sigma {sigma}")

    tracker.start_performance_track()
    cnn_accuracy, cnn_prediction, cnn_report = evaluate_image_model(cnn_model, test_set, config)
    tracker.stop_performance_track(f"CNN evaluation blur sigma {sigma}")

    # MC Dropout
    tracker.start_performance_track()
    mc_means_probabilities, mc_uncertainty = mc_dropout_prediction_img(cnn_model, test_set, config)
    tracker.stop_performance_track(f"CNN MC Dropout blur sigma {sigma}")

    plot_uncertainty_distribution(mc_uncertainty, f"CNN MC Dropout blur sigma {sigma}", mc_threshold, config)

    # Deep Ensemble
    tracker.start_performance_track()
    de_means_probabilities, de_uncertainty = deep_ensemble_prediction_img(train_simple_cnn, config, test_set, train_set_corrupted  )
    tracker.stop_performance_track(f"CNN Deep Ensemble prediction sigma {sigma}")

    plot_uncertainty_distribution(de_uncertainty, f"CNN Deep Ensemble prediction sigma {sigma}", de_threshold, config)

    # Grad-CAM
    tracker.start_performance_track()
    heatmaps = gradcam_img(cnn_model, test_set, config)
    tracker.stop_performance_track(f"CNN Grad-CAM sigma {sigma}")

    experiment_id = build_experiment_id(dataset_short, model_name, 'TC12', f"blur{sigma}")

    # Show grid for expert review
    plot_gradcam_sample(test_set, heatmaps, review_count, config, f"CNN {dataset_name} blur sigma {sigma}", save=True)
    consistency_scores, correct, incorrect, partial = collect_gradcam_feedback_resumable(
        review_count, config, experiment_id, batch_size=10
    )
    print(f"Human feedback: correct: {correct}, incorrect: {incorrect}, partial: {partial}")


    # Get true  labels for first samples
    y_true_reviewed = np.array([test_set[i][1] for i in range(review_count)])

    # MC Dropout with samples
    mc_y_pred_reviewed = mc_means_probabilities[:review_count].argmax(axis=1)
    mc_flags_reviewed = assign_flags(mc_uncertainty[:review_count], consistency_scores, mc_threshold, 0.5)
    mc_flag_result_reviewed = evaluate_flags(mc_flags_reviewed, mc_y_pred_reviewed, y_true_reviewed)
    plot_flag_distribution(mc_flags_reviewed, f"CNN MC + Grad-CAM blur {sigma}", config)


    # Deep Ensemble with samples
    de_y_pred_reviewed = de_means_probabilities[:review_count].argmax(axis=1)
    de_flags_reviewed = assign_flags(de_uncertainty[:review_count], consistency_scores, de_threshold, 0.5)
    de_flag_result_reviewed = evaluate_flags(de_flags_reviewed, de_y_pred_reviewed, y_true_reviewed)
    plot_flag_distribution(de_flags_reviewed, f"CNN DE + Grad-CAM blur {sigma}", config)


    # Sample test set flagging (UQ only, no human review)
    fake_consistency_reviewed = np.ones(review_count)

    # MC only
    mc_flags_uq_only = assign_flags(mc_uncertainty[:review_count], fake_consistency_reviewed, mc_threshold, 0.5)
    mc_20_result = evaluate_flags(mc_flags_uq_only, mc_y_pred_reviewed, y_true_reviewed)
    plot_flag_distribution(mc_flags_uq_only, "CNN MC UQ Only", config)

    # Deep Ensemble only
    de_flags_uq_only = assign_flags(de_uncertainty[:review_count], fake_consistency_reviewed, de_threshold, 0.5)
    de_20_result = evaluate_flags(de_flags_uq_only, de_y_pred_reviewed, y_true_reviewed)
    plot_flag_distribution(de_flags_uq_only, "CNN DE UQ Only", config)

    # Generate report
    level_result = {
    'perturbation': 'blur',
    'model': 'Simple CNN',
    'accuracy': round(cnn_accuracy, 4),
    'classification_report': cnn_report,
    'mc_uncertainty': {
        'mean': round(float(mc_uncertainty.mean()), 8),
        'max': round(float(mc_uncertainty.max()), 8),
        'threshold': mc_threshold
    },
    'de_uncertainty': {
        'mean': round(float(de_uncertainty.mean()), 8),
        'max': round(float(de_uncertainty.max()), 8),
        'threshold': de_threshold
    },
    'gradcam_feedback':{
        'correct': int(correct),
        'incorrect': int(incorrect),
        'partial': int(partial)
    },
    'flagging_mc': mc_flag_result_reviewed,
    'flagging_de': de_flag_result_reviewed,
    'flagging_mc_20_mc_only': mc_20_result,
    'flagging_mc_20_de_only': de_20_result,
    'mc_vs_mc_plus_xai': round(mc_flag_result_reviewed['GREEN']['accuracy'] - mc_20_result['GREEN']['accuracy'], 4),
    'de_vs_de_plus_xai': round(de_flag_result_reviewed['GREEN']['accuracy'] - de_20_result['GREEN']['accuracy'], 4),
    'performance': tracker.get_results()
    }

    blur_result[f"sigma_{sigma}"] = level_result

    save_per_sample(
        config, experiment_id,
        y_true=y_true_reviewed,
        y_pred=mc_y_pred_reviewed,
        mc_uncertainty=mc_uncertainty[:review_count],
        de_uncertainty=de_uncertainty[:review_count],
        consistency=consistency_scores,
    )

    report_output = generate_report(config, f"{dataset_name} - CNN blur sigma {sigma}", stage2_result=level_result)
    save_report(report_output, f'{dataset_short.upper()}_TC_12_Simple_CNN_Blur_Sigma{sigma}.json', config)

Simple CNN training started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
Simple CNN training completed.
CNN training blur sigma 1.0: Time:595.39s, Memory:700.91MB
 Image model evaluation started.
{'0': {'precision': 0.986159169550173, 'recall': 0.57, 'f1-score': 0.7224334600760456, 'support': 1000.0}, '1': {'precision': 0.4643478260869565, 'recall': 0.801, 'f1-score': 0.5878899082568807, 'support': 1000.0}, '2': {'precision': 0.7159090909090909, 'recall': 0.441, 'f1-score': 0.5457920792079208, 'support': 1000.0}, '3': {'precision': 0.6506986027944112, 'recall': 0.652, 'f1-score': 0.6513486513486514, 'support': 1000.0}, '4': {'precision': 0.7371794871794872, 'recall': 0.575, 'f1-score': 0.6460674157303371, 'support': 1000.0}, '5': {'precision': 0.30380517503805177, 'recall': 0.998, 'f1-score': 0.4658109684947491, 'support': 1000.0}, '6': {'precision': 0.42665388302972196, 'recall': 0.445, 'f1-score': 0.4356338717572198, 'support': 1000.0}, '7': {'precision': 0.8632478632478633, 'reca

In [5]:
brightness_result = {}

for delta in brightness_level:

    # Load corrupted training data
    train_set_corrupted = load_corrupted_image_data(config, 'brightness', delta)

    tracker.start_performance_track()
    cnn_model = train_simple_cnn(train_set_corrupted, config)
    tracker.stop_performance_track(f"CNN training brightness delta {delta}")

    tracker.start_performance_track()
    cnn_accuracy, cnn_prediction, cnn_report = evaluate_image_model(cnn_model, test_set, config)
    tracker.stop_performance_track(f"CNN evaluation brightness delta {delta}")

    # MC Dropout
    tracker.start_performance_track()
    mc_means_probabilities, mc_uncertainty = mc_dropout_prediction_img(cnn_model, test_set, config)
    tracker.stop_performance_track(f"CNN MC Dropout brightness delta {delta}")

    plot_uncertainty_distribution(mc_uncertainty, f"CNN MC Dropout brightness delta {delta}", mc_threshold, config)

    # Deep Ensemble
    tracker.start_performance_track()
    de_means_probabilities, de_uncertainty = deep_ensemble_prediction_img(train_simple_cnn, config, test_set, train_set_corrupted)
    tracker.stop_performance_track(f"CNN Deep Ensemble prediction delta {delta}")

    plot_uncertainty_distribution(de_uncertainty, f"CNN Deep Ensemble prediction delta {delta}", de_threshold, config)

    # Grad-CAM
    tracker.start_performance_track()
    heatmaps = gradcam_img(cnn_model, test_set, config)
    tracker.stop_performance_track(f"CNN Grad-CAM delta {delta}")

    experiment_id = build_experiment_id(dataset_short, model_name, 'TC12', f"bright{delta}")

    plot_gradcam_sample(test_set, heatmaps, review_count, config, f"CNN {dataset_name} brightness delta {delta}", save=True)
    consistency_scores, correct, incorrect, partial = collect_gradcam_feedback_resumable(
        review_count, config, experiment_id, batch_size=10
    )

    y_true_reviewed = np.array([test_set[i][1] for i in range(review_count)])

    # MC Dropout reviewed
    mc_y_pred_reviewed = mc_means_probabilities[:review_count].argmax(axis=1)
    mc_flags_reviewed = assign_flags(mc_uncertainty[:review_count], consistency_scores, mc_threshold, 0.5)
    mc_flag_result_reviewed = evaluate_flags(mc_flags_reviewed, mc_y_pred_reviewed, y_true_reviewed)
    plot_flag_distribution(mc_flags_reviewed, f"CNN MC + Grad-CAM brightness {delta}", config)

    # Deep Ensemble reviewed
    de_y_pred_reviewed = de_means_probabilities[:review_count].argmax(axis=1)
    de_flags_reviewed = assign_flags(de_uncertainty[:review_count], consistency_scores, de_threshold, 0.5)
    de_flag_result_reviewed = evaluate_flags(de_flags_reviewed, de_y_pred_reviewed, y_true_reviewed)
    plot_flag_distribution(de_flags_reviewed, f"CNN DE + Grad-CAM brightness {delta}", config)

    # UQ only on reviewed samples
    fake_consistency_reviewed = np.ones(review_count)

    mc_flags_uq_only = assign_flags(mc_uncertainty[:review_count], fake_consistency_reviewed, mc_threshold, 0.5)
    mc_20_result = evaluate_flags(mc_flags_uq_only, mc_y_pred_reviewed, y_true_reviewed)

    de_flags_uq_only = assign_flags(de_uncertainty[:review_count], fake_consistency_reviewed, de_threshold, 0.5)
    de_20_result = evaluate_flags(de_flags_uq_only, de_y_pred_reviewed, y_true_reviewed)

    # Generate report
    level_result = {
        'perturbation': 'brightness',
        'model': 'Simple CNN',
        'accuracy': round(cnn_accuracy, 4),
        'classification_report': cnn_report,
        'mc_uncertainty': {
            'mean': round(float(mc_uncertainty.mean()), 8),
            'max': round(float(mc_uncertainty.max()), 8),
            'threshold': mc_threshold
        },
        'de_uncertainty': {
            'mean': round(float(de_uncertainty.mean()), 8),
            'max': round(float(de_uncertainty.max()), 8),
            'threshold': de_threshold
        },
        'gradcam_feedback': {
            'correct': int(correct),
            'incorrect': int(incorrect),
            'partial': int(partial)
        },
        'flagging_mc': mc_flag_result_reviewed,
        'flagging_de': de_flag_result_reviewed,
        'flagging_mc_20_mc_only': mc_20_result,
        'flagging_mc_20_de_only': de_20_result,
        'mc_vs_mc_plus_xai': round(mc_flag_result_reviewed['GREEN']['accuracy'] - mc_20_result['GREEN']['accuracy'], 4),
        'de_vs_de_plus_xai': round(de_flag_result_reviewed['GREEN']['accuracy'] - de_20_result['GREEN']['accuracy'], 4),
        'performance': tracker.get_results()
    }

    brightness_result[f"delta_{delta}"] = level_result

    save_per_sample(
        config, experiment_id,
        y_true=y_true_reviewed,
        y_pred=mc_y_pred_reviewed,
        mc_uncertainty=mc_uncertainty[:review_count],
        de_uncertainty=de_uncertainty[:review_count],
        consistency=consistency_scores,
    )

    report_output = generate_report(config, f"{dataset_name} - CNN brightness delta {delta}", stage2_result=level_result)
    save_report(report_output, f'{dataset_short.upper()}_TC_12_Simple_CNN_Brightness_Delta{delta}.json', config)

Simple CNN training started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
Simple CNN training completed.
CNN training brightness delta 0.2: Time:359.09s, Memory:0.00MB
 Image model evaluation started.
{'0': {'precision': 0.9655172413793104, 'recall': 0.476, 'f1-score': 0.6376423308774279, 'support': 1000.0}, '1': {'precision': 0.22333243316230084, 'recall': 0.827, 'f1-score': 0.3516904103763555, 'support': 1000.0}, '2': {'precision': 0.7547169811320755, 'recall': 0.08, 'f1-score': 0.14466546112115733, 'support': 1000.0}, '3': {'precision': 0.8658892128279884, 'recall': 0.297, 'f1-score': 0.4422933730454207, 'support': 1000.0}, '4': {'precision': 1.0, 'recall': 0.034, 'f1-score': 0.06576402321083172, 'support': 1000.0}, '5': {'precision': 0.36867559523809523, 'recall': 0.991, 'f1-score': 0.5374186550976139, 'support': 1000.0}, '6': {'precision': 0.288135593220339, 'recall': 0.391, 'f1-score': 0.3317776834959695, 'support': 1000.0}, '7': {'precision': 0.9349005424954792, 'recall': 0.51

In [6]:
for i in range(10):
       print(f"[{i}] std={heatmaps[i].std():.4f}, max={heatmaps[i].max():.4f}")

[0] std=0.2093, max=1.0000
[1] std=0.1489, max=1.0000
[2] std=0.1299, max=1.0000
[3] std=0.3007, max=1.0000
[4] std=0.3455, max=1.0000
[5] std=0.0767, max=1.0000
[6] std=0.1192, max=1.0000
[7] std=0.3401, max=1.0000
[8] std=0.3305, max=1.0000
[9] std=0.3160, max=1.0000
